In [28]:
!git clone https://github.com/sinh2206/RL_final_project.git

fatal: destination path 'RL_final_project' already exists and is not an empty directory.


In [29]:
!pip install kaggle-environments

In [30]:
import os, sys, time, importlib, itertools, math
from collections import defaultdict
from pathlib import Path
import numpy as np
from kaggle_environments import make

In [31]:
def _field(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def run_single_game(agent1, agent2, seed=None, config=None):
    # Dam bao moi game chi co 500 steps
    cfg = {"seed": seed, "episodeSteps": 500}
    if config:
        cfg.update(config)
        # Khong cho phep ghi de sai episodeSteps
        cfg["episodeSteps"] = 500

    env = make("orbit_wars", configuration=cfg, debug=False)
    env.run([agent1, agent2])

    final_step = env.steps[-1]
    statuses = [_field(s, "status", None) for s in final_step]
    rewards = [_field(s, "reward", 0) for s in final_step]

    errors = {}
    for i, status in enumerate(statuses):
        if status != "DONE":
            errors[i] = status  # "ERROR", "TIMEOUT", "INVALID", ...

    if errors:
        return None, errors

    r0 = 0.0 if rewards[0] is None else float(rewards[0])
    r1 = 0.0 if rewards[1] is None else float(rewards[1])

    if r0 > r1:
        return 0, None
    elif r1 > r0:
        return 1, None
    else:
        return -1, None

In [32]:
def _stable_seed(name1, name2, game_idx):
    # Deterministic seed independent of Python hash randomization
    a = sum((i + 1) * ord(c) for i, c in enumerate(name1))
    b = sum((i + 1) * ord(c) for i, c in enumerate(name2))
    return int((a * 1000003 + b * 9176 + game_idx * 7919) % (2**31 - 1))


def run_match(agent1, agent2, name1, name2, n_games=50, config=None):
    wins = [0, 0]
    draws = 0
    errors = []

    for game in range(n_games):
        seed = _stable_seed(name1, name2, game)
        winner, err = run_single_game(agent1, agent2, seed=seed, config=config)

        if err:
            errors.append((game, err))
            # Dung som khi gap loi
            return wins, draws, errors

        if winner == 0:
            wins[0] += 1
        elif winner == 1:
            wins[1] += 1
        else:
            draws += 1

    return wins, draws, errors

In [33]:
# Tu dong tim thu muc RL_agent (tranh loi duong dan tuong doi)
def _resolve_agent_dir():
    candidates = [
        Path('RL_agent'),
        Path('RL_final_project') / 'RL_agent',
        Path('/content/RL_final_project/RL_agent'),
        Path('/content/RL_agent'),
        Path('/content/RL_project/RL_agent'),
        Path('/content/RL_final_project/RL_final_project/RL_agent'),
        Path.cwd() / 'RL_agent',
        Path.cwd() / 'RL_final_project' / 'RL_agent',
    ]

    # Bo trung va giu thu tu
    uniq = []
    seen = set()
    for c in candidates:
        try:
            rc = c.resolve()
        except Exception:
            rc = c
        key = str(rc)
        if key not in seen:
            seen.add(key)
            uniq.append(c)

    for c in uniq:
        try:
            if c.exists() and c.is_dir():
                return c
        except Exception:
            pass
    return None


AGENT_DIR = _resolve_agent_dir()
agents = {}  # name -> callable or package path

print('Working dir :', Path.cwd())
print('Agent dir   :', AGENT_DIR)

if AGENT_DIR is None:
    print('ERROR: Khong tim thay thu muc RL_agent. Hay kiem tra cell clone/git pull.')


def _validate_agent_spec(agent_spec, seed=12345):
    """
    Smoke test de xac nhan spec nay la 1 agent co the chay.
    Tra ve (ok, message)
    """
    try:
        winner, err = run_single_game(agent_spec, 'random', seed=seed)
        if err:
            # Van la 1 agent, nhung bi loi luc chay
            return True, f'runtime status={err}'
        return True, 'ok'
    except Exception as e:
        return False, str(e)


if AGENT_DIR is not None:
    py_files = sorted(AGENT_DIR.glob('*.py'))
    tgz_files = sorted(AGENT_DIR.glob('*.tar.gz'))

    print(f'Found {len(py_files)} .py and {len(tgz_files)} .tar.gz files in {AGENT_DIR}')
    for f in py_files[:10]:
        print(' -', f.name)
    if len(py_files) > 10:
        print(' ...')

    # 1) Load agent .py
    for file in py_files:
        if file.name.startswith('_'):
            continue

        try:
            spec = importlib.util.spec_from_file_location(file.stem, str(file))
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
        except Exception as e:
            print(f'WARNING: Loi import {file.name}: {e}')
            continue

        if hasattr(module, 'agent') and callable(module.agent):
            agents[file.stem] = module.agent
        else:
            print(f'WARNING: {file} khong co ham agent()')

    # 2) Load agent package .tar.gz
    for file in tgz_files:
        name = file.name[:-7]  # bo duoi .tar.gz
        spec_path = str(file.resolve())

        ok, msg = _validate_agent_spec(spec_path)
        if ok:
            agents[name] = spec_path
            print(f'Loaded tar.gz agent: {file.name} ({msg})')
        else:
            print(f'WARNING: {file.name} khong phai agent hop le: {msg}')

print(f'Da load {len(agents)} agent: {list(agents.keys())}')

Working dir : /content
Agent dir   : RL_final_project/RL_agent
Found 5 .py and 0 .tar.gz files in RL_final_project/RL_agent
 - agent_PPO_v1.py
 - agent_rule_base_v1.py
 - agent_rule_base_v2.py
 - agent_rule_base_v3.py
 - agent_rule_base_v4.py
Da load 5 agent: ['agent_PPO_v1', 'agent_rule_base_v1', 'agent_rule_base_v2', 'agent_rule_base_v3', 'agent_rule_base_v4']


In [34]:
disqualified = set()  # agent bi loai vi loi
error_log = []        # ghi chi tiet loi
valid_agents = set(agents.keys())

In [35]:
results = []
total_wins = defaultdict(int)

rule_group = sorted([n for n in valid_agents if 'agent_rule_base' in n])
other_group = sorted([n for n in valid_agents if 'agent_rule_base' not in n])

print('Nhom 1 (rule_base):', rule_group)
print('Nhom 2 (other):', other_group)

# Lich dau:
# - Nhom 2 dau noi bo
# - Nhom 2 dau voi Nhom 1 (1 luot cho moi cap)
# - Khong co tran Nhom 1 vs Nhom 1
match_pairs = []
match_pairs.extend(list(itertools.combinations(other_group, 2)))
match_pairs.extend([(a2, a1) for a2 in other_group for a1 in rule_group])

if not match_pairs:
    print('Khong co cap dau hop le theo quy tac chia nhom.')
else:
    print(f'Tong so cap dau theo lich: {len(match_pairs)}')

for name1, name2 in match_pairs:
    if name1 in disqualified or name2 in disqualified:
        continue

    if name1 not in agents or name2 not in agents:
        print(f'SKIP: thieu agent {name1} hoac {name2}')
        continue

    agent1 = agents[name1]
    agent2 = agents[name2]

    print(f'Dang dau: {name1} vs {name2}')
    wins, draws, errors = run_match(agent1, agent2, name1=name1, name2=name2, n_games=50)

    if errors:
        faulty_agents = set()
        for game_idx, err_dict in errors:
            for agent_idx, status in err_dict.items():
                faulty = name1 if agent_idx == 0 else name2
                faulty_agents.add(faulty)
                disqualified.add(faulty)
                error_log.append(
                    f'Loi {status} o game {game_idx} giua {name1} va {name2}: {faulty}'
                )
        print(f"  -> Loi: {', '.join(sorted(faulty_agents))} bi loai.")
        continue

    total_wins[name1] += wins[0]
    total_wins[name2] += wins[1]

    score1 = wins[0] / 50.0
    score2 = wins[1] / 50.0

    results.append({
        'match': f'{name1} vs {name2}',
        'score': f'{score1:.2f} vs {score2:.2f}',
        'wins': wins,
        'draws': draws,
    })

    print(f'  Ket qua: {wins[0]} - {wins[1]} (hoa {draws})')

Dang dau: agent_PPO_v1 vs agent_rule_base_v1
  Ket qua: 7 - 43 (hoa 0)
Dang dau: agent_PPO_v1 vs agent_rule_base_v2
  Ket qua: 1 - 49 (hoa 0)
Dang dau: agent_PPO_v1 vs agent_rule_base_v3
  Ket qua: 13 - 37 (hoa 0)
Dang dau: agent_PPO_v1 vs agent_rule_base_v4
  Ket qua: 48 - 2 (hoa 0)
Dang dau: agent_rule_base_v1 vs agent_rule_base_v2


KeyboardInterrupt: 

In [ ]:
with open("results.txt", "w", encoding="utf-8") as f:
    f.write("KET QUA GIAI DAU ORBIT WARS")
    f.write("\n")
    f.write("============================")
    f.write("\n\n")

    if results:
        for r in results:
            f.write(f"Van dau({r['match']}), ti so({r['score']}), hoa({r['draws']})")
            f.write("\n")
    else:
        f.write("Chua co tran hop le nao duoc ghi nhan.")
        f.write("\n")

    f.write("\nTONG SO TRAN THANG CUA MOI AGENT:\n")
    for name in sorted(valid_agents):
        if name not in disqualified:
            f.write(f"{name}: {total_wins[name]} thang")
            f.write("\n")

    f.write("\nAGENT BI LOI:\n")
    if error_log:
        for err in error_log:
            f.write(err)
            f.write("\n")
    else:
        f.write("Khong co loi.\n")

print("Da ghi ket qua vao results.txt")

Da ghi ket qua vao results.txt


In [ ]:
print()
print("=== KET QUA ===")
if results:
    for r in results:
        print(f"{r['match']}: {r['score']} (hoa {r['draws']})")
else:
    print("Khong co ket qua hop le nao.")

print()
print("Tong thang:")
for name in sorted(valid_agents):
    if name not in disqualified:
        print(f"{name}: {total_wins[name]}")

if disqualified:
    print()
    print("Agent bi loai vi loi:")
    for name in sorted(disqualified):
        print(name)

if error_log:
    print()
    print("Chi tiet loi:")
    for err in error_log:
        print("-", err)


=== KET QUA ===
Khong co ket qua hop le nao.

Tong thang:
